# Age and Gender Prediction using ResNet50 Transfer Learning

This notebook builds an end-to-end **Age + Gender Prediction** model using the **UTKFace dataset** and **ResNet50** as a transfer learning backbone.

The model has two outputs:

1. **Age prediction**: regression output
2. **Gender prediction**: binary classification output

This notebook is written from scratch and uses a modern `tf.data` pipeline to avoid old Keras generator errors such as:

```text
TypeError: output_signature must contain objects that are subclass of tf.TypeSpec
```


## Cell 1: Import Required Libraries

This cell imports all important libraries used for dataset handling, image preprocessing, model building, training, evaluation, and prediction.


In [ ]:
import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

print("TensorFlow Version:", tf.__version__)


## Cell 2: Kaggle Secret Key Method

Use this cell in Google Colab.

Before running it, add these two secrets in Colab:

```text
KAGGLE_USERNAME
KAGGLE_KEY
```

Go to: **Left sidebar → Secrets icon → Add new secret**.


In [ ]:
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

print("Kaggle credentials loaded from Colab secrets.")


## Cell 3: Download UTKFace Dataset from Kaggle

This cell downloads the UTKFace dataset.

UTKFace image names contain labels in this format:

```text
age_gender_race_datetime.jpg
```

Example:

```text
25_0_0_201701.jpg.chip.jpg
```

Here:

- `25` = age
- `0` = male
- `1` = female


In [ ]:
!kaggle datasets download -d jangedoo/utkface-new -p /content --force


## Cell 4: Extract Dataset

This cell extracts the downloaded ZIP file into the `/content` directory.


In [ ]:
zip_path = "/content/utkface-new.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully.")


## Cell 5: Set Dataset Folder Path

The aligned cropped UTKFace images are usually stored inside:

```text
/content/utkface_aligned_cropped/UTKFace
```


In [ ]:
folder_path = Path("/content/utkface_aligned_cropped/UTKFace")

print("Folder exists:", folder_path.exists())
print("Total files:", len(list(folder_path.glob("*.jpg"))))


## Cell 6: Create DataFrame from Image Filenames

This cell reads image filenames and extracts:

- Age
- Gender
- Image path

Any corrupted or incorrectly named files are skipped.


In [ ]:
records = []

for img_file in folder_path.glob("*.jpg"):
    try:
        parts = img_file.name.split("_")
        age = int(parts[0])
        gender = int(parts[1])

        if gender in [0, 1] and 1 <= age <= 116:
            records.append({
                "image_path": str(img_file),
                "age": age,
                "gender": gender
            })
    except Exception:
        continue

df = pd.DataFrame(records)

print("Dataset shape:", df.shape)
df.head()


## Cell 7: Check Age and Gender Distribution

This helps us understand whether the dataset is balanced or imbalanced.


In [ ]:
print("Gender distribution:")
print(df["gender"].value_counts())

plt.figure(figsize=(8, 4))
plt.hist(df["age"], bins=30)
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Number of Images")
plt.show()


## Cell 8: Split Data into Training and Validation Sets

We use 80% data for training and 20% for validation.


In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=df["gender"]
)

print("Training samples:", train_df.shape)
print("Validation samples:", val_df.shape)


## Cell 9: Create TensorFlow Dataset Pipeline

This modern `tf.data` pipeline avoids the old `ImageDataGenerator` multi-output problem.

For each image, it returns:

```python
image, {"age": age_label, "gender": gender_label}
```

This structure exactly matches our model outputs.


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE


def load_image_and_labels(image_path, age, gender):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = preprocess_input(image)

    labels = {
        "age": tf.cast(age, tf.float32),
        "gender": tf.cast(gender, tf.float32)
    }

    return image, labels


def create_dataset(dataframe, shuffle=False):
    image_paths = dataframe["image_path"].values
    ages = dataframe["age"].values
    genders = dataframe["gender"].values

    dataset = tf.data.Dataset.from_tensor_slices((image_paths, ages, genders))
    dataset = dataset.map(load_image_and_labels, num_parallel_calls=AUTOTUNE)

    if shuffle:
        dataset = dataset.shuffle(buffer_size=1000)

    dataset = dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return dataset


train_ds = create_dataset(train_df, shuffle=True)
val_ds = create_dataset(val_df, shuffle=False)

print("TensorFlow datasets created successfully.")


## Cell 10: Visualize Sample Images

This cell displays a few sample images with their age and gender labels.

Remember:

- `0 = Male`
- `1 = Female`


In [ ]:
for images, labels in train_ds.take(1):
    plt.figure(figsize=(12, 8))
    for i in range(9):
        img = images[i].numpy()
        # Reverse ResNet preprocessing approximately for display
        img = img[..., ::-1]
        img = img + np.array([103.939, 116.779, 123.68])
        img = np.clip(img, 0, 255).astype("uint8")

        age = int(labels["age"][i].numpy())
        gender = int(labels["gender"][i].numpy())
        gender_name = "Female" if gender == 1 else "Male"

        plt.subplot(3, 3, i + 1)
        plt.imshow(img)
        plt.title(f"Age: {age}, Gender: {gender_name}")
        plt.axis("off")
    plt.show()


## Cell 11: Build ResNet50 Transfer Learning Model

We use **ResNet50 pretrained on ImageNet**.

Important points:

- `include_top=False` removes the original ImageNet classifier.
- `base_model.trainable=False` freezes ResNet50 layers.
- We add our own custom heads for age and gender.


In [ ]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation="relu")(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)

age_output = layers.Dense(1, activation="linear", name="age")(x)
gender_output = layers.Dense(1, activation="sigmoid", name="gender")(x)

model = Model(inputs=inputs, outputs={"age": age_output, "gender": gender_output})

model.summary()


## Cell 12: Compile the Model

The model uses two losses:

1. **Age loss**: Mean Absolute Error because age is a regression problem.
2. **Gender loss**: Binary Crossentropy because gender is binary classification.

We also use loss weights so that gender learning is not ignored.


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss={
        "age": "mae",
        "gender": "binary_crossentropy"
    },
    metrics={
        "age": ["mae"],
        "gender": ["accuracy"]
    },
    loss_weights={
        "age": 1.0,
        "gender": 5.0
    }
)


## Cell 13: Define Callbacks

Callbacks help improve training:

- **EarlyStopping** stops training if validation loss stops improving.
- **ModelCheckpoint** saves the best model.
- **ReduceLROnPlateau** reduces learning rate when training becomes stuck.


In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "best_resnet50_age_gender_model.keras",
    monitor="val_loss",
    save_best_only=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    min_lr=1e-7
)


## Cell 14: Train the Model

This trains only the custom top layers while ResNet50 remains frozen.

Start with 10 epochs. You can increase it later.


In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop, checkpoint, reduce_lr]
)


## Cell 15: Plot Training Curves

This cell plots age error and gender accuracy.


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history["age_mae"], label="Train Age MAE")
plt.plot(history.history["val_age_mae"], label="Validation Age MAE")
plt.title("Age Prediction MAE")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history.history["gender_accuracy"], label="Train Gender Accuracy")
plt.plot(history.history["val_gender_accuracy"], label="Validation Gender Accuracy")
plt.title("Gender Prediction Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


## Cell 16: Fine-Tuning ResNet50

After training the custom head, we unfreeze the last few ResNet50 layers and train with a very small learning rate.

Fine-tuning can improve performance.


In [ ]:
base_model.trainable = True

# Freeze earlier layers and fine-tune only last 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss={
        "age": "mae",
        "gender": "binary_crossentropy"
    },
    metrics={
        "age": ["mae"],
        "gender": ["accuracy"]
    },
    loss_weights={
        "age": 1.0,
        "gender": 5.0
    }
)

fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[early_stop, checkpoint, reduce_lr]
)


## Cell 17: Evaluate the Model

This cell evaluates the final model on validation data.


In [ ]:
results = model.evaluate(val_ds)
print("Evaluation Results:", results)


## Cell 18: Predict Age and Gender on One Image

Upload or give the path of any face image and run this function.


In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array


def predict_age_gender(image_path):
    img = load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = img_to_array(img)
    img_array = preprocess_input(img_array)
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)

    age = prediction["age"][0][0]
    gender_prob = prediction["gender"][0][0]
    gender = "Female" if gender_prob >= 0.5 else "Male"

    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Predicted Age: {age:.1f}, Gender: {gender}")
    plt.show()

    print(f"Predicted Age: {age:.1f}")
    print(f"Predicted Gender: {gender}")
    print(f"Gender Probability Female: {gender_prob:.3f}")


# Example:
# predict_age_gender('/content/sample_face.jpg')


## Cell 19: Save Final Model

This saves the final trained model.


In [ ]:
model.save("resnet50_age_gender_final_model.keras")
print("Model saved successfully.")


## Cell 20: Load Saved Model

Use this cell when you want to reuse the trained model later.


In [ ]:
loaded_model = tf.keras.models.load_model("resnet50_age_gender_final_model.keras")
print("Model loaded successfully.")


# Summary

In this notebook, we built an end-to-end age and gender prediction system using:

- UTKFace dataset
- Kaggle secret-key method
- TensorFlow `tf.data` pipeline
- ResNet50 transfer learning
- Multi-output model
- Age regression
- Gender classification
- Fine-tuning
- Prediction function
- Model saving and loading
